In [38]:
import numpy as np 
import pandas as pd  

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, brier_score_loss


from ufc_betting.config import settings
from ufc_betting.Models.LogisticRegression.train_test_builder import TrainTestBuilder


In [41]:
# Edit the input file, split dates, and selected_feats for your experiment.
fp = settings.all_feats_dir / 'entire_odds_stats_2026-10-04.csv'
df_model = pd.read_csv(fp)

ODDS_TYPE = 'open'
VAL_START_DATE = '2024-05-11'
TEST_START_DATE = '2026-02-21'

# Supply raw dataframe column names here (before encoding or polynomial expansion).
selected_feats = [
    'proba_fair_open_diff',
    'age_diff',
    'td_attempted_pm_diff',
    'win_streak_diff',

    'adjusted_td_red',
    'adjusted_td_blue',

    'adjusted_sig_str_blue',
    'adjusted_sig_str_red',

    'kd_total_diff',
    'td_landed_pm_diff',
    'td_attempted_pm_red',
    'td_attempted_pm_blue',
    'adjusted_control_red',
    'adjusted_control_blue',

    'leg_str_pm_red',
    'leg_str_pm_blue',
    'months_since_diff',

    'ratio_sig_str_red',
    'ratio_sig_str_blue',

    'opp_age_red', 'opp_age_blue',

    'reach_diff',
    'sub_att_pm_red',
    'sub_att_pm_blue',
    'ratio_control_diff',
    'td_landed_pm_diff',
    'ratio_td_diff',

    'sig_str_absorbed_total_diff',
    'sig_str_accuracy_pct_diff',
    'sig_str_defense_pct_diff',

    'win_pct_red',
    'win_pct_blue',
    
    'lose_streak_diff',
    'elo_red',
    'elo_blue',
    'elo_pred',
    'womens_fight',
    'age_red',
    'age_blue',
]


def prepare_feature_data(df, feats, *, odds_type=ODDS_TYPE,
                         val_start_date=VAL_START_DATE,
                         test_start_date=TEST_START_DATE):
    """Build train/validation/test inputs from any nonempty feature list."""
    if isinstance(feats, str):
        raise ValueError('feats must be a list of column names, not a string')
    feats = list(dict.fromkeys(feats))
    if not feats:
        raise ValueError('Pass at least one feature in feats')
    builder = TrainTestBuilder(
        df=df, feats=feats, target_col='winner', date_col='date',
        odds_type=odds_type, year=2010, month=2, day=26,
        required_features_only=True,
    )
    return builder.prepare_train_test(
        val_start_date=val_start_date, test_start_date=test_start_date,
        scale=True, save_split=False,
    )


# Or call prepare_feature_data(df_model, your_own_feats).
split = prepare_feature_data(df_model, selected_feats)
X_train, y_train = split['X_train'], split['y_train']
X_val, y_val = split['X_val'], split['y_val']
X_test, y_test = split['X_test'], split['y_test']
df_train, df_val, df_test = split['train_df'], split['val_df'], split['test_df']
scaler, encoder = split['scaler'], split['encoder']

# Scaling/encoding is fitted on training rows only. Both searches below are
# evaluated on validation; X_test/y_test remain reserved for final evaluation.
# Cap k at the input width so even degree=1 works with a short custom list.
selection_sizes = sorted({min(k, X_train.shape[1]) for k in (25, 50, 75)}) + ['all']
display(split['split_summary'])
print(f'{len(selected_feats)} requested features; {X_train.shape[1]} encoded inputs')


Filtered: kept 4903 rows from 2010-02-26 onward.
PREPARE SHAPE: (4903, 303)
train: 3,961 rows (80.79%) | 2010-03-21 to 2024-05-04
val: 664 rows (13.54%) | 2024-05-11 to 2026-02-07
test: 278 rows (5.67%) | 2026-02-21 to 2026-10-03


,split,rows,percentage,first_date,last_date
0,train,3961,80.787273,2010-03-21,2024-05-04
1,val,664,13.542729,2024-05-11,2026-02-07
2,test,278,5.669998,2026-02-21,2026-10-03


39 requested features; 38 encoded inputs


In [42]:
X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 3961 entries, 0 to 3960
Data columns (total 38 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   proba_fair_open_diff         3961 non-null   float64
 1   age_diff                     3961 non-null   float64
 2   td_attempted_pm_diff         3961 non-null   float64
 3   win_streak_diff              3961 non-null   float64
 4   adjusted_td_red              3961 non-null   float64
 5   adjusted_td_blue             3961 non-null   float64
 6   adjusted_sig_str_blue        3961 non-null   float64
 7   adjusted_sig_str_red         3961 non-null   float64
 8   kd_total_diff                3961 non-null   float64
 9   td_landed_pm_diff            3961 non-null   float64
 10  td_attempted_pm_red          3961 non-null   float64
 11  td_attempted_pm_blue         3961 non-null   float64
 12  adjusted_control_red         3961 non-null   float64
 13  adjusted_control_blue        

In [43]:
# Expand all inputs, keeping categorical interactions but no categorical powers.
# TrainTestBuilder's encoder identifies every one-hot categorical input.
categorical_cols = (
    encoder.get_feature_names_out().tolist() if encoder is not None else []
)
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
poly.fit(X_train)
categorical_input_mask = X_train.columns.isin(categorical_cols)
keep_poly_terms = ~(poly.powers_[:, categorical_input_mask] > 1).any(axis=1)
poly_feature_names = poly.get_feature_names_out(X_train.columns)[keep_poly_terms]


def transform_polynomial_features(X, fitted_poly=poly, keep_terms=keep_poly_terms):
    return pd.DataFrame(
        fitted_poly.transform(X.loc[:, fitted_poly.feature_names_in_])[:, keep_terms],
        columns=fitted_poly.get_feature_names_out()[keep_terms],
        index=X.index,
    )


X_poly = transform_polynomial_features(X_train)
X_poly_val = transform_polynomial_features(X_val)
X_poly_test = transform_polynomial_features(X_test)
print(X_poly.shape)


(3961, 777)


In [44]:
# Select numeric columns
numeric_cols = X_poly.select_dtypes(include=np.number).columns
variances = X_poly[numeric_cols].var()
variance_ranking = variances.sort_values(ascending=False)

print(variance_ranking.head(5))
print(variance_ranking.shape)

sub_att_pm_blue^2                            299.754084
sub_att_pm_red^2                              69.216888
td_attempted_pm_blue^2                        56.503702
months_since_diff^2                           44.727330
td_attempted_pm_diff td_attempted_pm_blue     34.723254
dtype: float64
(777,)


In [47]:
from mifs import MutualInformationFeatureSelector

# categorical=True tells mifs that y contains class labels.
# The package does not support a separate categorical mask for X.
feat_selector = MutualInformationFeatureSelector(
    method='MRMR', n_features=50, categorical=True, n_jobs=-1,
)
feat_selector.fit(X_poly, y_train.astype(np.int64))
X_selected = feat_selector.transform(X_poly)


c:\Users\jcmar\my_files\SportsBetting\venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but MutualInformationFeatureSelector was fitted without feature names
  warnings.warn(


In [48]:
selected_columns = X_poly.columns[feat_selector._support_mask]
print(selected_columns)
print(len(selected_columns))

Index(['proba_fair_open_diff adjusted_control_red',
       'proba_fair_open_diff leg_str_pm_blue',
       'proba_fair_open_diff ratio_sig_str_blue',
       'proba_fair_open_diff opp_age_red', 'proba_fair_open_diff elo_blue',
       'age_diff adjusted_sig_str_red', 'age_diff ratio_control_diff',
       'age_diff sig_str_defense_pct_diff',
       'td_attempted_pm_diff adjusted_sig_str_red',
       'td_attempted_pm_diff age_red', 'win_streak_diff ratio_sig_str_red',
       'adjusted_td_red opp_age_red', 'adjusted_td_red win_pct_blue',
       'adjusted_td_red age_blue', 'adjusted_td_blue kd_total_diff',
       'adjusted_td_blue months_since_diff',
       'adjusted_td_blue ratio_sig_str_red', 'adjusted_td_blue win_pct_red',
       'adjusted_sig_str_blue td_attempted_pm_red',
       'adjusted_sig_str_blue leg_str_pm_blue',
       'adjusted_sig_str_blue elo_blue', 'adjusted_sig_str_blue age_blue',
       'kd_total_diff sig_str_accuracy_pct_diff',
       'td_landed_pm_diff adjusted_control_blu

In [63]:
# Simple logistic regression on the polynomial features.
X_poly_val = transform_polynomial_features(X_val)
poly_logit = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(
        penalty='elasticnet',
        solver='saga',
        C=0.5,
        l1_ratio=0.5,
        max_iter=5000,
        random_state=42,
    )),
])
poly_logit.fit(X_poly[selected_columns[:10]], y_train)
y_poly_val_pred = poly_logit.predict(X_poly_val[selected_columns[:10]])
print(f'val accuracy: {accuracy_score(y_val, y_poly_val_pred):.4f}')
print(f'val brier score: {brier_score_loss(y_val, y_poly_val_pred):.4f}')

val accuracy: 0.5467
val brier score: 0.4533


c:\Users\jcmar\my_files\SportsBetting\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [ ]:
pipe = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scale_poly", StandardScaler()),
    ("select", SelectKBest(score_func=f_classif)),
    ("model", LogisticRegression(
        penalty="l1",
        solver="saga",
        max_iter=5000
    ))
])


param_grid = {
    # overall polynomial degree
    "poly__degree": [1, 2, 3],

    # whether to include interactions
    "poly__interaction_only": [True, False],

    # how many transformed features to keep
    "select__k": selection_sizes,
    "model__C": [3.5]
}


grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=10,
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

poly = best_model.named_steps["poly"]
selector = best_model.named_steps["select"]
                                  
feature_names = poly.get_feature_names_out(
    X_train.columns
)

mask = selector.get_support()
selected_features = feature_names[mask]

print(selected_features)
print(len(selected_features))
print(feature_names)

In [ ]:
print(grid.best_params_)
print(grid.best_score_)

{'model__C': 3.5, 'poly__degree': 1, 'poly__interaction_only': False, 'select__k': 20}
0.6998162795015099


: 

In [ ]:
# Validation metrics (the final test split is reserved).

y_pred = best_model.predict(X_val)

acc = accuracy_score(
    y_val,
    y_pred
)

print(acc)

In [ ]:
pipe = Pipeline([
    ("spline", SplineTransformer(include_bias=False)),
    ("scale_spline", StandardScaler()),
    ("select", SelectKBest(score_func=f_classif)),
    (
        "model",
        LogisticRegression(
            penalty="l1",          # lasso penalty
            solver="saga",         # required for l1
            max_iter=5000
        )
    )
])


param_grid = {
    # spline degree
    "spline__degree": [4,2,3],

    # number of knots
    "spline__n_knots": [2, 3, 5],

    # knot placement
    "spline__knots": ["uniform", "quantile"],

    # number of transformed spline features to keep
    "select__k": selection_sizes,
    "model__C": [3.5]
}


grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=10,
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train, y_train)

# best fitted pipeline
best_model = grid.best_estimator_

# pipeline steps
spline = best_model.named_steps["spline"]
selector = best_model.named_steps["select"]

# get spline-transformed feature names
feature_names = spline.get_feature_names_out(
    X_train.columns
)

# boolean mask from SelectKBest
mask = selector.get_support()

# selected spline features
selected_features = feature_names[mask]

print(selected_features)
print(len(selected_features))



In [ ]:
print(grid.best_params_)
print(grid.best_score_)

In [ ]:
# Validation metrics (the final test split is reserved).
y_pred = best_model.predict(X_val)
y_proba = best_model.predict_proba(X_val)

acc = accuracy_score(
    y_val,
    y_pred
)

print(acc)
print(brier_score_loss(y_val, y_proba[:,1]))



In [ ]:
# Validation metrics (the final test split is reserved).
lr = LogisticRegression(max_iter=1000, penalty='l1', solver="saga", C=5.5)
lr.fit(X_train, y_train)
y_hat = lr.predict(X_val)
y_proba = lr.predict_proba(X_val)

y_hat_train = lr.predict(X_train)

print(accuracy_score(y_val,y_hat))
print(accuracy_score(y_train, y_hat_train))
print(brier_score_loss(y_val, y_proba[:,1]))